# NeuroDermAI Kaggle Training Notebook

This notebook trains a real transfer-learning image classifier for common skin conditions using TensorFlow/Keras on Kaggle GPU.

Canonical classes:
- `acne`
- `eczema`
- `psoriasis`
- `fungal`
- `warts`
- `normal`

How it works:
1. Reads image data from `/kaggle/input/...`
2. Maps dataset folder names into the canonical class set when safe aliases are found
3. Builds a clean directory at `/kaggle/working/curated_dataset/`
4. Trains a MobileNetV2 classifier with augmentation and class weights
5. Saves `/kaggle/working/model.h5` and `/kaggle/working/labels.json`

If the attached dataset does not include healthy images, the exported model will not include the `normal` class. That limitation is intentional and should be documented honestly in the app README.

In [ ]:
import json
import re
import shutil
from collections import Counter
from pathlib import Path
from uuid import uuid4

import numpy as np
import tensorflow as tf

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2
INITIAL_EPOCHS = 8
FINE_TUNE_EPOCHS = 5
MIN_IMAGES_PER_CLASS = 20

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
DATASET_DIR = None

TARGET_CLASSES = ["acne", "eczema", "psoriasis", "fungal", "warts", "normal"]
CLASS_ALIASES = {
    "acne": ["acne", "acne vulgaris", "pimple", "pimples"],
    "eczema": ["eczema", "atopic dermatitis"],
    "psoriasis": ["psoriasis", "psoriatic"],
    "fungal": ["fungal", "fungal infection", "ringworm", "tinea", "candidiasis"],
    "warts": ["warts", "wart", "verruca"],
    "normal": ["normal", "healthy", "clear skin", "no disease"],
}
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

tf.random.set_seed(SEED)
np.random.seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))


def normalise_name(raw_name: str) -> str:
    cleaned = re.sub(r"[^a-z0-9]+", " ", raw_name.lower())
    return re.sub(r"\s+", " ", cleaned).strip()


def canonical_class_name(raw_name: str):
    normalized = normalise_name(raw_name)
    for canonical, aliases in CLASS_ALIASES.items():
        if normalized == canonical:
            return canonical
        for alias in aliases:
            alias_normalized = normalise_name(alias)
            if alias_normalized and alias_normalized in normalized:
                return canonical
    return None


def discover_dataset_dir() -> Path:
    if DATASET_DIR:
        dataset_path = Path(DATASET_DIR)
        if not dataset_path.exists():
            raise FileNotFoundError(f"Configured DATASET_DIR does not exist: {dataset_path}")
        return dataset_path

    candidates = [path for path in KAGGLE_INPUT_ROOT.iterdir() if path.is_dir()]
    if not candidates:
        raise FileNotFoundError("No attached Kaggle datasets were found under /kaggle/input.")

    if len(candidates) == 1:
        return candidates[0]

    scored_candidates = []
    for candidate in candidates:
        score = 0
        for subdir in candidate.rglob("*"):
            if subdir.is_dir() and canonical_class_name(subdir.name):
                score += 1
        scored_candidates.append((score, candidate))

    scored_candidates.sort(key=lambda item: item[0], reverse=True)
    print("Autodetected dataset candidates:")
    for score, path in scored_candidates[:5]:
        print(f"  score={score}: {path}")

    best_score, best_candidate = scored_candidates[0]
    if best_score == 0:
        raise ValueError(
            "Could not infer a matching dataset from /kaggle/input. Set DATASET_DIR to the correct folder."
        )
    return best_candidate


def infer_class_from_path(image_path: Path, dataset_dir: Path):
    relative_parts = image_path.relative_to(dataset_dir).parts[:-1]
    for part in reversed(relative_parts):
        canonical = canonical_class_name(part)
        if canonical:
            return canonical
    return None


def build_curated_dataset(source_dir: Path):
    curated_dir = WORKING_DIR / "curated_dataset"
    if curated_dir.exists():
        shutil.rmtree(curated_dir)
    curated_dir.mkdir(parents=True, exist_ok=True)

    counts = Counter()

    for image_path in source_dir.rglob("*"):
        if not image_path.is_file() or image_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue

        canonical = infer_class_from_path(image_path, source_dir)
        if canonical is None:
            continue

        destination_dir = curated_dir / canonical
        destination_dir.mkdir(parents=True, exist_ok=True)

        parent_stub = normalise_name(str(image_path.relative_to(source_dir).parent)).replace(" ", "_")
        destination_name = f"{parent_stub}_{uuid4().hex[:10]}{image_path.suffix.lower()}"
        shutil.copy2(image_path, destination_dir / destination_name)
        counts[canonical] += 1

    if not counts:
        raise ValueError(
            "No labeled images matched the canonical class aliases. Attach a dataset whose folder names include supported labels."
        )

    available_counts = {}
    for class_name in TARGET_CLASSES:
        class_dir = curated_dir / class_name
        image_count = counts.get(class_name, 0)

        if image_count == 0:
            if class_dir.exists():
                shutil.rmtree(class_dir)
            continue

        if image_count < MIN_IMAGES_PER_CLASS:
            print(
                f"Skipping '{class_name}' because only {image_count} images were found, below MIN_IMAGES_PER_CLASS={MIN_IMAGES_PER_CLASS}."
            )
            shutil.rmtree(class_dir)
            continue

        available_counts[class_name] = image_count

    if len(available_counts) < 2:
        raise ValueError(
            "Training requires at least two usable classes after dataset curation."
        )

    return curated_dir, available_counts


source_dir = discover_dataset_dir()
curated_dir, class_counts = build_curated_dataset(source_dir)

print(f"Using source dataset: {source_dir}")
print(f"Curated dataset written to: {curated_dir}")
print("Usable class counts:")
for class_name, count in class_counts.items():
    print(f"  {class_name}: {count}")

if "normal" not in class_counts:
    print(
        "NOTE: No usable 'normal' class was found. The exported model will only predict the available disease classes."
    )


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

AUTOTUNE = tf.data.AUTOTUNE
TOP_K = min(3, len(class_counts))

raw_train_ds = keras.utils.image_dataset_from_directory(
    curated_dir,
    validation_split=VALIDATION_SPLIT,
    subset="training",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
)

raw_val_ds = keras.utils.image_dataset_from_directory(
    curated_dir,
    validation_split=VALIDATION_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
)

class_names = raw_train_ds.class_names
print("Training classes:", class_names)

augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.12),
        layers.RandomContrast(0.12),
    ],
    name="augmentation",
)


def prepare_train(images, labels):
    images = tf.cast(images, tf.float32)
    images = augmentation(images, training=True)
    images = preprocess_input(images)
    return images, labels


def prepare_eval(images, labels):
    images = tf.cast(images, tf.float32)
    images = preprocess_input(images)
    return images, labels


train_ds = raw_train_ds.map(prepare_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = raw_val_ds.map(prepare_eval, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

total_images = sum(class_counts[class_name] for class_name in class_names)
class_weight = {
    index: round(total_images / (len(class_names) * class_counts[class_name]), 4)
    for index, class_name in enumerate(class_names)
}
print("Class weights:", class_weight)

base_model = MobileNetV2(
    input_shape=IMAGE_SIZE + (3,),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = keras.Input(shape=IMAGE_SIZE + (3,), name="image")
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.35)(x)
outputs = layers.Dense(len(class_names), activation="softmax", dtype="float32")(x)
model = keras.Model(inputs, outputs, name="neurodermai_mobilenetv2")

metrics = [
    keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
    keras.metrics.SparseTopKCategoricalAccuracy(k=TOP_K, name="top_3_accuracy"),
]

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=metrics,
)

checkpoint_path = WORKING_DIR / "best_model.keras"
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6),
]

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks,
)

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=metrics,
)

history_fine_tune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS + FINE_TUNE_EPOCHS,
    initial_epoch=history_head.epoch[-1] + 1,
    class_weight=class_weight,
    callbacks=callbacks,
)

best_model = keras.models.load_model(checkpoint_path)
val_probabilities = best_model.predict(val_ds, verbose=0)
val_true = np.concatenate([labels.numpy() for _, labels in raw_val_ds], axis=0)
val_pred = np.argmax(val_probabilities, axis=1)

print(classification_report(val_true, val_pred, target_names=class_names, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(val_true, val_pred))

labels_payload = {
    "class_names": class_names,
    "image_size": list(IMAGE_SIZE),
    "backbone": "MobileNetV2",
    "validation_split": VALIDATION_SPLIT,
}

model_path = WORKING_DIR / "model.h5"
labels_path = WORKING_DIR / "labels.json"

best_model.save(model_path)
with labels_path.open("w", encoding="utf-8") as handle:
    json.dump(labels_payload, handle, indent=2)

print(f"Saved model to: {model_path}")
print(f"Saved labels to: {labels_path}")


In [ ]:
import matplotlib.pyplot as plt

sample_images, sample_labels = next(iter(raw_val_ds.take(1)))
sample_image = sample_images[0].numpy().astype("uint8")
ground_truth = class_names[int(sample_labels[0].numpy())]

sample_input = preprocess_input(tf.cast(sample_images[:1], tf.float32))
sample_probabilities = best_model.predict(sample_input, verbose=0)[0]
top_indices = sample_probabilities.argsort()[::-1][:TOP_K]

print(f"Ground truth: {ground_truth}")
print("Top predictions:")
for index in top_indices:
    print(f"  {class_names[index]}: {float(sample_probabilities[index]):.4f}")

plt.figure(figsize=(4, 4))
plt.imshow(sample_image)
plt.axis("off")
plt.title(f"Ground truth: {ground_truth}")
plt.show()
